> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# v6.7 ·  The same.  GPT‑5.5， Change it to...  8B  Text tool protocol for 

 Selection  **CPU runtime**， On the left.  Secrets  For Ben.  notebook  Open Existing  `OPENAI_API_KEY`  Access rights, more.  **Runtime → Run all**。 First check pairing input, run  8  Article  smoke， Then run automatically  384  The initial budget is separate from this supplemental experiment.  **US$15  Cumulative ceiling **。 Time off for reaching the upper limit or for reporting a service error.  v6.5  and  v6.6  Call all reuses. 

Drive  Directory As  `MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p7_text_protocol_bridge96/bridge-main`。 Use same after interruption  notebook  Again.  Run all。 Use one active run session. If a previous request was sent without a saved receipt, first verify  API  Well use it and decide if were going to...  RETRY_UNCERTAIN  Set  True。 Do not use the old  runtime  Take over the writing lock while still working.  ZIP  Reread the check, Drive  Auto-release after sync is complete  runtime。


In [ ]:
#@title  Run Settings 
RUN_MODE = "all"  #@param ["all","smoke","export"]
USD_BUDGET = 15.0  #@param {type:"number"}
RETRY_UNCERTAIN = False  #@param {type:"boolean"}
TAKEOVER_LOCK_ID = ""  #@param {type:"string"}
AUTO_DISCONNECT = True  #@param {type:"boolean"}
import base64, hashlib, json, os, zlib
from pathlib import Path
from google.colab import drive
if RUN_MODE not in ("all","smoke","export") or not 0 < USD_BUDGET <= 15:raise ValueError(" Legality of stage or budget ")
MOUNT67=Path("/content/drive")
drive.mount(str(MOUNT67))
if not os.path.ismount(str(MOUNT67)) or not (MOUNT67/"MyDrive").is_dir():raise RuntimeError(" Please finish.  Drive  Mount Rerun ")
RUN67=MOUNT67/"MyDrive/AAAI_Workshop/agent_entrainment/agent_entrainment_v6p7_text_protocol_bridge96/bridge-main"
RESULT67=None
print(" Table of contents of the current cycle ",RUN67," Cumulative budget  USD",USD_BUDGET)


# v6.7  Same model text  JSON  Tool Contrast 

 This supplemental experiment is being viewed.  v6.5  and  v6.6  Designed as a result. Reuse the same generator in the formal phase.  confirm-00097  to  confirm-00192、 The same target, initial stock, and task records. Each world runs preview / Preferred  ×  Check it out first.  X/ Check it out first.  Y  Four new tracks, in total.  384  Article. 2  Independent.  pilot  The world.  8  trajectories only validate interfaces and estimate cost. There is no natural starting condition. Run each condition once and analyze at the world level. v6.5   384  Article does not interfere  8B  Text trajectories with  v6.6   384  Article  GPT  Original controlled tracks are read and reused only. 

 new group with  v6.6  Use same  GPT-5.5  A snapshot.  gpt-5.5-2026-04-23、Responses API、medium  Logic, default sampling, standard service lines and each  4096  output  token  upper limit. Every request passes.  input  Strings sent for freezing  v6.5  Pure Text  Actor  Hints, with 10 examples, actual tasks, full visual tool history and  NEXT_TOOL_CALL。 No native function registered, no server used  JSON schema、 Previous Response  ID  Or step-by-step reasoning. Press  8B  The original parser takes one.  JSON  objects; allow one enclosing code fence, but do not repair fields. Execute the action with the original tools and rerender text for the next step. The limit is  6  actions, including the fixed first lookup actually executed by the program. Effects of prompt roles, examples, history representation, parsing, and reasoning continuation cannot be isolated individually. 

 The only main comparison is  G(GPT Text )  Less  G(GPT Native )。 Each world first uses the same label.  Y  Less first check.  X  Final actual purchase  Y  metric, then subtract preview from preferred. Also report the wrong-lookup risk difference averaged equally over both labels in the same model  R  the difference between (even)  G  Zero can also change the whole thing.  GPT  and  8B   G  differences, strict success after correct first lookups, and label-specific rechecks, incorrect-order attempts, and actual  Y  orders, fallback, other items, and no orders. Format failures, refusals, truncation, and no orders remain in the denominator and are reported separately. All three groups use the same final-order scoring. After collecting  96  The effects are calculated after a complete world, and the world is paired and re-sampled.  3000  Second, Seeds  76102， Double  95%  Percent area. A full-time World Class. Any controlled branch buys it.  Y” One side  95%  upper bound, while also checking no orders and correct completion. These are paired protocol diagnostics on known worlds. 


In [ ]:
SOURCES67={}
ASSETS67={}


In [ ]:
#@title sources/01_shared.py
SOURCES67['sources/01_shared.py']=r'''import copy, hashlib, json

def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)

def stable_hash(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()

def _json_no_duplicates(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result

class OrderEnvironment:
    """A local, deterministic world; it knows inventory, not customer truth."""

    def __init__(self, initial_state):
        self.state = copy.deepcopy(initial_state)
        self.state.setdefault("stock", {})
        self.state.setdefault("reservations", {})
        self.state.setdefault("orders", [])
        self.state.setdefault("next_reservation", 1)
        self.state.setdefault("next_order", 1)
        if any(type(v) is not int or v < 0 for v in self.state["stock"].values()):
            raise ValueError("stock quantities must be nonnegative integers")
        self.tool_log = []

    def snapshot(self):
        return copy.deepcopy(self.state)

    def step(self, action):
        observation = self._execute(action)
        self.tool_log.append({"action": copy.deepcopy(action), "observation": copy.deepcopy(observation)})
        return copy.deepcopy(observation)

    def _execute(self, action):
        if not isinstance(action, dict):
            return {"ok": False, "error": "action_must_be_object"}
        tool = action.get("tool")
        schemas = {
            "check_stock": {"tool", "item"},
            "reserve_item": {"tool", "item"},
            "create_order": {"tool", "customer", "item", "city", "reservation"},
        }
        if not isinstance(tool, str) or tool not in schemas:
            return {"ok": False, "error": "unknown_tool"}
        if set(action) != schemas[tool] or any(not isinstance(v, str) or not v for v in action.values()):
            return {"ok": False, "error": "invalid_arguments", "tool": tool}
        item = action["item"]
        if item not in self.state["stock"]:
            return {"ok": False, "error": "unknown_item", "item": item}
        if tool == "check_stock":
            return {"ok": True, "item": item, "available": self.state["stock"][item]}
        if tool == "reserve_item":
            if self.state["stock"][item] <= 0:
                return {"ok": False, "error": "out_of_stock", "item": item}
            reservation = f"R{self.state['next_reservation']:04d}"
            self.state["next_reservation"] += 1
            self.state["stock"][item] -= 1
            self.state["reservations"][reservation] = {"item": item, "consumed": False}
            return {"ok": True, "item": item, "reservation": reservation}
        reservation = self.state["reservations"].get(action["reservation"])
        if reservation is None or reservation["consumed"] or reservation["item"] != item:
            return {"ok": False, "error": "invalid_reservation"}
        # No oracle city/customer correction: a semantically wrong order can
        # succeed as a tool call. The held-out evaluator scores it afterwards.
        order = {"order_id": f"O{self.state['next_order']:04d}",
                 "customer": action["customer"], "item": item,
                 "city": action["city"], "reservation": action["reservation"]}
        self.state["next_order"] += 1
        self.state["orders"].append(order)
        reservation["consumed"] = True
        return {"ok": True, "order": copy.deepcopy(order)}

GX_ATTRIBUTES = ("delivery_city", "order_customer", "preferred_item")

GX_ORDER_FIELDS = {
    "delivery_city": "city", "order_customer": "customer", "preferred_item": "item",
}

def gx_task_payload(case):
    """Only the other two fixed fields and fallback accompany the request ID."""
    return {"request_id": case["request_id"], "resolve_attribute": case["attribute"],
            "fixed_fields": {attribute: case["gold_fields"][attribute]
                             for attribute in GX_ATTRIBUTES if attribute != case["attribute"]},
            "fallback_item": case["fallback_item"]}

def gx_expected_order(case, initial_state=None):
    """Evaluator-only final fields, including stock-contingent SKU truth."""
    state = case["initial_env"] if initial_state is None else initial_state
    fields = case["gold_fields"]
    preferred = fields["preferred_item"]
    item = preferred if state["stock"].get(preferred, 0) > 0 else case["fallback_item"]
    return {"city": fields["delivery_city"], "customer": fields["order_customer"], "item": item}

def gx_value_missing(value):
    """Absent/null/blank values are not explicit choices of a wrong value.

    Nonstring values are schema/type failures, not choices of a string-valued
    attribute. They are excluded from wrong-choice rates by gx_wrong_value;
    the schema-complete indicators still expose their invalidity.
    """
    return value is None or (isinstance(value, str) and not value.strip())

def gx_wrong_value(value, expected):
    """Count only an explicitly supplied, nonblank string that is incorrect."""
    return isinstance(value, str) and bool(value.strip()) and value != expected

def gx_order_schema_complete(action):
    """A complete order attempt has exactly five keys with nonblank strings.

    This is a diagnostic; the original executable schema remains unchanged.
    It deliberately treats a whitespace-only value as missing even if the
    executable tool's nonempty-string check would accept that value.
    """
    return (isinstance(action, dict)
            and set(action) == {"tool", "customer", "item", "city", "reservation"}
            and action.get("tool") == "create_order"
            and all(isinstance(value, str) and bool(value.strip()) for value in action.values()))

def gx_score_actor(case, events, orders, initial_state):
    """Score accepted fields and the executed prerequisite subsequence.

    A wrong focal field is counted on any accepted order, independently of the
    correctness of the other fields. Missing orders are failures and have their
    own indicator; they never masquerade as correct field decisions. Tool
    recovery is allowed: strict success requires a valid final check/reserve/
    order subsequence, while all earlier tool errors remain reported separately.

    Intermediate rates are source-level booleans, not counts of repeated calls.
    The first order attempt is the first parsed create_order action, including
    a rejected one; any-attempt metrics cover all such actions. Missing or
    nonstring focal values do not count as explicit wrong choices. Reservation
    acceptance requires that reserve_item's own observation has ok=True.
    Expected items include the legal initial-stock fallback. Conflict-SKU
    reservation metrics apply only to preferred_item sources (None otherwise).
    """
    expected = gx_expected_order(case, initial_state)
    focal = GX_ORDER_FIELDS[case["attribute"]]
    correct = len(orders) == 1 and all(orders[0].get(key) == value for key, value in expected.items())
    preferred = case["gold_fields"]["preferred_item"]
    chosen = expected["item"]
    checked, grounded = {}, set()
    checked_preferred = checked_chosen = reserved_chosen = path_valid = False
    tool_errors = []
    actions = []
    order_attempts, reservation_attempts, accepted_reservations = [], [], []
    for step, event in enumerate(events):
        action = event.get("action")
        observation = event.get("observation")
        if isinstance(action, dict):
            actions.append(action)
            if action.get("tool") == "create_order":
                order_attempts.append(action)
            elif action.get("tool") == "reserve_item":
                reservation_attempts.append(action)
                if isinstance(observation, dict) and observation.get("ok") is True:
                    accepted_reservations.append(action)
        if not isinstance(action, dict) or not isinstance(observation, dict):
            continue
        if not observation.get("ok", False):
            tool_errors.append({"step": step, "error": observation.get("error")})
            continue
        tool, item = action.get("tool"), action.get("item")
        if tool == "check_stock":
            checked[item] = observation["available"]
            checked_preferred |= item == preferred
            checked_chosen |= item == chosen and observation["available"] > 0
        elif tool == "reserve_item" and item == chosen:
            reserved_chosen = True
            priority_ok = (checked.get(preferred, -1) > 0 if chosen == preferred
                           else checked.get(preferred, -1) == 0)
            if priority_ok and checked.get(chosen, 0) > 0:
                grounded.add(observation["reservation"])
        elif tool == "create_order" and item == chosen:
            path_valid |= action.get("reservation") in grounded
    protocol_errors = {"action_must_be_object", "unknown_tool", "invalid_arguments"}
    format_failure = any(event.get("parse_error") is not None for event in events) or any(
        error["error"] in protocol_errors for error in tool_errors)
    first_order = order_attempts[0] if order_attempts else None
    order_fields = ("customer", "item", "city", "reservation")
    first_missing_attribute = first_order is not None and gx_value_missing(first_order.get(focal))
    first_missing_city = first_order is not None and gx_value_missing(first_order.get("city"))
    first_missing_required = first_order is not None and any(
        gx_value_missing(first_order.get(field)) for field in order_fields)
    conflict_item_applicable = case["attribute"] == "preferred_item"
    return {
        "task_success": bool(correct), "strict_task_success": bool(correct and path_valid),
        "accepted_wrong_attribute": any(order.get(focal) != expected[focal] for order in orders),
        "accepted_conflict_value": any(order.get(focal) == case["conflict_value"] for order in orders),
        "accepted_correct_attribute": any(order.get(focal) == expected[focal] for order in orders),
        "accepted_wrong_other_fields": any(any(order.get(key) != value for key, value in expected.items()
                                                if key != focal) for order in orders),
        "no_accepted_order": not orders, "accepted_order_count": len(orders),
        "format_failure": bool(format_failure), "tool_error": bool(tool_errors),
        "tool_error_count": len(tool_errors), "action_count": len(actions),
        "order_attempted": first_order is not None,
        "order_accepted": bool(orders), "order_attempt_count": len(order_attempts),
        "first_order_attribute_correct": first_order is not None and first_order.get(focal) == expected[focal],
        "first_order_conflict_value": first_order is not None and first_order.get(focal) == case["conflict_value"],
        "any_order_conflict_value": any(action.get(focal) == case["conflict_value"] for action in order_attempts),
        "first_order_wrong_attribute": first_order is not None and gx_wrong_value(first_order.get(focal), expected[focal]),
        "any_order_wrong_attribute": any(gx_wrong_value(action.get(focal), expected[focal]) for action in order_attempts),
        "first_order_missing_attribute": bool(first_missing_attribute),
        "any_order_missing_attribute": any(gx_value_missing(action.get(focal)) for action in order_attempts),
        "first_order_missing_city": bool(first_missing_city),
        "any_order_missing_city": any(gx_value_missing(action.get("city")) for action in order_attempts),
        "first_order_missing_required_fields": bool(first_missing_required),
        "any_order_missing_required_fields": any(any(gx_value_missing(action.get(field)) for field in order_fields)
                                                  for action in order_attempts),
        "first_order_schema_complete": first_order is not None and gx_order_schema_complete(first_order),
        "any_order_schema_complete": any(gx_order_schema_complete(action) for action in order_attempts),
        "reservation_attempted": bool(reservation_attempts),
        "reservation_accepted": bool(accepted_reservations),
        "reservation_attempt_count": len(reservation_attempts),
        "accepted_reservation_count": len(accepted_reservations),
        "attempted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)
                                                 for action in reservation_attempts),
        "accepted_wrong_reservation_item": any(gx_wrong_value(action.get("item"), chosen)
                                                for action in accepted_reservations),
        "attempted_conflict_reservation_item": (
            any(action.get("item") == case["conflict_value"] for action in reservation_attempts)
            if conflict_item_applicable else None),
        "accepted_conflict_reservation_item": (
            any(action.get("item") == case["conflict_value"] for action in accepted_reservations)
            if conflict_item_applicable else None),
        "prerequisite_path_valid": bool(path_valid),
        "checked_preferred": bool(checked_preferred), "checked_chosen_available": bool(checked_chosen),
        "reserved_chosen": bool(reserved_chosen),
        "subgoal_progress": (int(checked_preferred) + int(checked_chosen) + int(reserved_chosen) + int(correct)) / 4,
    }

TEMPLATES = {
    "smoke": ("Process request {request_id}. Place exactly one order using its resolved fields.",),
    "pilot": ("Fulfil request {request_id} by placing one order using its resolved fields.",
              "Arrange one shipment for request {request_id} using its resolved order details."),
    "confirm": ("Submit one order for request {request_id} using the fields belonging to that request.",
                "Complete the purchase specified by request {request_id} using its recorded details."),
}

def case_for(w, target, stocked=True):
    other = 1 - target
    stock = {sku: 2 for sku in (*w["items"], w["fallback"])}
    stock[w["items"][target]] = 2 if stocked else 0
    return {
        "source_id": w["id"], "stage": w["split"], "attribute": "preferred_item",
        "request_id": w["requests"][target], "template_id": str(w["template"]),
        "template_text": TEMPLATES[w["split"]][w["template"]],
        "gold_fields": {"preferred_item": w["items"][target], "delivery_city": w["city"],
                        "order_customer": w["customer"]},
        "conflict_value": w["items"][other], "fallback_item": w["fallback"],
        "profile_event_id": w["events"][target], "note_event_id": w["events"][other],
        "stock_available": stocked, "initial_env": {"stock": stock, "orders": [], "reservations": {},
                                                   "next_reservation": 1, "next_order": 1},
    }

def records_for(w, target, context):
    if context not in ("C0", "C1") or target not in (0, 1):
        raise ValueError("Invalid target/context")
    records = []
    for index in range(2):
        has_item = index == target or context == "C1"
        records.append({"kind": "verified_profile", "entity": w["requests"][index],
                        "attribute": "preferred_item" if has_item else "dispatch_batch",
                        "value": w["items"][index] if has_item else w["lots"][index],
                        "source_event": w["events"][index]})
    return list(reversed(records)) if w["reverse"] else records

v6_records_for = records_for

def records_for(world, target, condition):
    if condition != "Clex":
        return v6_records_for(world, target, condition)
    records = v6_records_for(world, target, "C1")
    other_id = world["requests"][1-target]
    next(r for r in records if r["entity"] == other_id)["attribute"] = "preview_item"
    return records

NATIVE_POLICY = "You are an order-processing assistant. Execute one order using the tools.\nA request_id identifies a request, not a customer account. Resolve the indicated attribute for that request from its records, then combine it with the task's fixed_fields. delivery_city maps to order.city; order_customer maps to order.customer; preferred_item is the SKU to check and reserve when available. The fallback_item is allowed only if the resolved preferred SKU is out of stock.\nThe records belong to one system. Match the request ID and the indicated attribute. Actions and tool observations do not update the request fields.\nCheck the preferred SKU's stock. If unavailable, check the fallback. Reserve the chosen available SKU, then create exactly one order. Copy the reservation ID only from its successful reserve_item observation; a stock count is not a reservation.\nUse the provided functions, one tool call at a time."
'''


In [ ]:
#@title sources/02_receipts.py
SOURCES67['sources/02_receipts.py']=r'''"""Native tool replication. No weights, hooks or Hub calls; only the three local order tools."""
import copy
import csv
import hashlib
import io
import json
import math
import os
import re
import tempfile
import time
import zipfile
from pathlib import Path

MODEL66 = "gpt-5.5-2026-04-23"
ENDPOINT66 = "https://api.openai.com/v1/responses"
SCOPE66 = "v6p6_native_behavior96"
CONTEXTS66 = ("Clex","C1")
STARTS66 = ("natural","gold","foil")
PRICES66 = {"input":5.0,"cached":0.50,"output":30.0}  # USD per million tokens (standard short context)
PARAMS66 = dict(model=MODEL66,reasoning={"effort":"medium"},max_output_tokens=4096,
                include=["reasoning.encrypted_content"],
                parallel_tool_calls=False,tool_choice="auto",store=False,service_tier="default")


def functions66():
    spec = (("check_stock","Check available inventory for a SKU.",{"item":"SKU to check."}),
            ("reserve_item","Reserve one available item and return its reservation ID.",{"item":"SKU to reserve."}),
            ("create_order","Create an order using an active reservation for the item.",
             {"customer":"Customer account ID.","item":"SKU to order.","city":"Delivery city.",
              "reservation":"Reservation ID returned by reserve_item."}))
    return [{"type":"function","name":name,"description":desc,"strict":True,"parameters":{
            "type":"object","properties":{k:{"type":"string","description":v} for k,v in fields.items()},
            "required":list(fields),"additionalProperties":False}} for name,desc,fields in spec]


def visible66(w,t,c):
    case=case_for(w,t)
    content=(case["template_text"].format(request_id=case["request_id"])+"\nTASK_JSON:\n"+
             canonical_json(gx_task_payload(case))+"\nMEMORY_JSON:\n"+canonical_json(records_for(w,t,c)))
    return [{"role":"developer","content":NATIVE_POLICY},{"role":"user","content":content}]


def cells66(w):
    pairs=[(c,l) for c in CONTEXTS66 for l in STARTS66]
    return sorted(pairs,key=lambda pair:stable_hash(["v66-order-v1",w["id"],*pair]))


def atomic66(path,data):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    fd,name=tempfile.mkstemp(dir=path.parent,prefix=".pending-",suffix=".tmp")
    try:
        with os.fdopen(fd,"wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        for attempt in range(6):
            try:
                os.replace(name,path)
                break
            except PermissionError:
                if attempt==5:raise
                time.sleep(.04*2**attempt)
    finally:
        Path(name).unlink(missing_ok=True)


def save66(path,value):
    atomic66(path,canonical_json(value).encode())


def price66(usage):
    pin,pout=usage["input_tokens"],usage["output_tokens"]
    cached=(usage.get("input_tokens_details") or {}).get("cached_tokens",0)
    if any(type(v)!=int or v<0 for v in (pin,pout,cached)) or cached>pin:
        raise ValueError("Invalid API usage")
    return ((pin-cached)*PRICES66["input"]+cached*PRICES66["cached"]+pout*PRICES66["output"])/1e6


def native_action66(calls):
    if len(calls)!=1:
        raise ValueError("Expected one native tool call")
    call=calls[0]
    if call.get("type")!="function_call" or not isinstance(call.get("call_id"),str) or not call["call_id"]:
        raise ValueError("Invalid native call")
    args=json.loads(call["arguments"],object_pairs_hook=_json_no_duplicates)
    if not isinstance(args,dict) or "tool" in args:
        raise ValueError("Invalid native arguments")
    return {"tool":call["name"],**args},call["call_id"]


def messages66(w,t,c,start):
    env=OrderEnvironment(case_for(w,t)["initial_env"])
    messages=visible66(w,t,c)
    events=[]
    if start in ("gold","foil"):
        sku=w["items"][t if start=="gold" else 1-t]
        action={"tool":"check_stock","item":sku}
        obs=env.step(action)
        if obs!={"ok":True,"item":sku,"available":2}:raise ValueError("Unexpected fixed lookup")
        # Same opaque call id for both labels and both forced starts. No correctness annotations enter messages.
        cid="call_"+stable_hash(["v66",w["id"],t])[:24]
        messages+= [{"type":"function_call","call_id":cid,"name":"check_stock","arguments":canonical_json({"item":sku})},
                    {"type":"function_call_output","call_id":cid,"output":canonical_json(obs)}]
        events.append(dict(step=0,origin="fixed_prefix",action=action,observation=obs,parse_error=None))
    elif start!="natural": raise ValueError("Unknown start")
    return env,messages,events


def choice66(reply):
    """Keep complete output items for stateless continuation and receipt-level replay."""
    if reply["model"]!=MODEL66:
        raise RuntimeError(" The returned model is not a frozen snapshot. Please check the model.  API  Access rights. ")
    status=reply["status"]
    reason=(reply.get("incomplete_details") or {}).get("reason")
    if status not in ("completed","incomplete") or (status=="incomplete" and reason not in ("max_output_tokens","content_filter")):
        raise RuntimeError("Responses API  No scorable terminal state was returned. Verify that request status and cost before resuming. ")
    if not isinstance(reply["output"],list):raise ValueError("Invalid Responses output")
    return dict(model=reply["model"],response_id=reply["id"],status=status,incomplete_reason=reason,
                service_tier=reply.get("service_tier"),output=reply["output"],usage=reply["usage"])


def play66(w,t,c,start,get_receipt):
    env,messages,events=messages66(w,t,c,start)
    step=0;terminal="action_budget";stopped_text=""
    while len(events)<6:
        got=get_receipt(step,dict(PARAMS66,input=messages,tools=functions66()))
        step+=1
        output=got["output"]
        parts=[part for item in output if item["type"]=="message" for part in item["content"]]
        stopped_text="\n".join(part["text"] for part in parts if part["type"]=="output_text")
        if got["status"]=="incomplete":
            terminal="length" if got["incomplete_reason"]=="max_output_tokens" else "content_filter"
            break
        refusals=[part["refusal"] for part in parts if part["type"]=="refusal"]
        if refusals:
            terminal="explicit_refusal";stopped_text="\n".join(refusals);break
        calls=[item for item in output if item["type"]=="function_call"]
        if not calls:
            terminal="text_reply" if stopped_text else "no_tool_call";break
        try:action,cid=native_action66(calls)
        except (TypeError,ValueError,KeyError):
            events.append(dict(step=len(events),origin="model",action=None,observation=None,parse_error="malformed_native_tool_call"))
            terminal="format_failure";break
        obs=env.step(action)
        events.append(dict(step=len(events),origin="model",action=action,observation=obs,parse_error=None))
        # Each returned item stays in order, including encrypted reasoning and assistant message phase.
        messages.extend(output)
        messages.append({"type":"function_call_output","call_id":cid,"output":canonical_json(obs)})
        if obs.get("error") in ("action_must_be_object","unknown_tool","invalid_arguments"):
            terminal="protocol_failure";break
        if action["tool"]=="create_order" and obs["ok"]:
            terminal="order_created";break
    return dict(events=events,final_state=env.snapshot(),terminal=terminal,stop_text=stopped_text)


class OpenAI66:
    def __init__(self,key):
        import requests
        self.session=requests.Session()
        self.session.headers.update({"Authorization":"Bearer "+key,"Content-Type":"application/json"})

    def __call__(self,payload):
        try:
            response=self.session.post(ENDPOINT66,json=payload,timeout=(15,180),allow_redirects=False)
        except Exception:
            raise RuntimeError(" The network was interrupted. API  Whether the check is charged is unknown. Please rerun to check the checking request. ") from None
        if response.status_code in (400,401,403,404,422,429):
            try: code=str(response.json().get("error",{}).get("code") or "")
            except ValueError: code=""
            safe=code if re.fullmatch(r"[A-Za-z0-9_.-]{1,80}",code) else ""
            raise Rejected66(f"API  Request denied.  (HTTP {response.status_code} {safe})。 Check keys, model privileges, balances or limit stream follow-up. ")
        if response.status_code!=200:
            raise RuntimeError(f"API  Status  {response.status_code}， The cost is to be reconciled. ")
        return response.json()


class Rejected66(RuntimeError):
    """The server explicitly rejected the request before returning a completion."""


class Study66:
    def __init__(self,root,inputs,smoke,reference,assets,source_sha,provider,budget,retry_unknown=False):
        self.root,self.inputs,self.smoke,self.reference=Path(root),inputs,smoke,reference
        self.assets,self.provider,self.budget,self.retry_unknown=assets,provider,budget,retry_unknown
        if not isinstance(budget,(int,float)) or isinstance(budget,bool) or not math.isfinite(budget) or budget<=0:
            raise ValueError("USD_BUDGET  Must be positive. ")
        self.manifest=dict(scope=SCOPE66,endpoint=ENDPOINT66,model=MODEL66,parameters=PARAMS66,tool_schemas=functions66(),
              assets_sha256={k:hashlib.sha256(v).hexdigest() for k,v in assets.items()},source_sha256=source_sha,
              policy=NATIVE_POLICY,pricing_per_million=PRICES66,max_actions=6,bootstrap_seed=76102,bootstrap_draws=3000)
        self.sig=stable_hash(self.manifest)
        manifest=self.root/"manifest.json"
        if manifest.is_file():
            if stable_hash(json.loads(manifest.read_bytes()))!=self.sig: raise RuntimeError(" Tabs are tied to other protocols. Use new protocols.  RUN_LABEL。")
        elif self.root.is_dir() and any(self.root.iterdir()):raise RuntimeError(" Missing directory  manifest， Please use a new label. ")
        else: save66(manifest,self.manifest)
        self.states={}
        for stage in ("smoke","full"):
            for p in (self.root/stage).glob("*/*.json"):
                name=p.relative_to(self.root).as_posix()[:-5]
                doc=json.loads(p.read_bytes())
                if doc["signature"]!=stable_hash([self.sig,name]) or stable_hash(doc["state"])!=doc["digest"]:
                    raise RuntimeError(" Checkpoint validation failed: "+name)
                self.states[name]=doc["state"]

    def worlds(self,stage):
        if stage=="full":return self.inputs["worlds"],self.inputs["target_by_world"]
        if stage=="smoke":return self.smoke["worlds"],self.smoke["target_by_world"]
        raise ValueError("RUN_STAGE  Only for...  smoke  or  full")

    def persist(self,key,state):
        save66(self.root/(key+".json"),{"signature":stable_hash([self.sig,key]),"digest":stable_hash(state),"state":state})
        self.states[key]=state

    def usage(self,stage=None):
        calls=0;known=uncertain=0.0;counts=[0,0,0]
        for key,s in self.states.items():
            if stage and not key.startswith(stage+"/"):continue
            for call in s["calls"]:
                calls+=1
                u=call["receipt"]["usage"]
                known+=price66(u);counts[0]+=u["input_tokens"];counts[1]+=u["output_tokens"]
                counts[2]+=(u.get("input_tokens_details") or {}).get("cached_tokens",0)
            uncertain+=sum(x["reserve_usd"] for x in s["uncertain"])
            if s["pending"]:uncertain+=s["pending"]["reserve_usd"]
        return dict(calls=calls,estimated_usd=known,uncertain_reserve_usd=uncertain,
                    input_tokens=counts[0],output_tokens=counts[1],cached_input_tokens=counts[2])

    def receipt(self,key,state,step,payload):
        request_hash=stable_hash(payload)
        if step<len(state["calls"]):
            old=state["calls"][step]
            if old["request_hash"]!=request_hash:raise RuntimeError(" The recovery is not consistent with the history of the request: "+key)
            return old["receipt"]
        if state["pending"]:
            if state["pending"]["request_hash"]!=request_hash:raise RuntimeError(" The history of requests to be reconciled does not match ")
            if not self.retry_unknown:
                raise RuntimeError(" Last request completed unknown: "+key+"。 Check  API  If the amount is accepted for possible double-counting, it will  RETRY_UNCERTAIN=True  Keep going. ")
            state["uncertain"].append(state["pending"]);state["pending"]=None
            self.persist(key,state)
        # Operational reserve uses UTF-8 request bytes, framing allowance and the configured maximum output.
        reserve=((len(canonical_json(payload).encode())+2048)*PRICES66["input"]+
                 PARAMS66["max_output_tokens"]*PRICES66["output"])/1e6
        meter=self.usage()
        if meter["estimated_usd"]+meter["uncertain_reserve_usd"]+reserve>self.budget:
            raise RuntimeError(" The local budget gate was reached. Checkpoints are saved; verify usage and raise the limit to continue  USD_BUDGET。")
        state["pending"]=dict(request_hash=request_hash,reserve_usd=reserve)
        self.persist(key,state)
        try:
            raw=self.provider(payload)
        except Rejected66:
            state["pending"]=None;self.persist(key,state)
            raise
        got=choice66(raw)
        price66(got["usage"])
        state["calls"].append(dict(request_hash=request_hash,receipt=got))
        state["pending"]=None
        self.persist(key,state)
        return got

    def trajectory(self,stage,w,t,c,start):
        key=f"{stage}/{w['id']}/{t}-{c}-{start}"
        state=self.states.get(key)
        if state and state.get("result") is not None:return state["result"]
        if state is None:
            state=dict(world=w["id"],target=t,context=c,start=start,calls=[],pending=None,uncertain=[],result=None)
        result=play66(w,t,c,start,lambda i,p:self.receipt(key,state,i,p))
        state["result"]=result
        self.persist(key,state)
        return result

    def completed(self,stage):
        worlds,ts=self.worlds(stage)
        return sum(self.states.get(f"{stage}/{w['id']}/{ts[w['id']]}-{c}-{l}",{}).get("result") is not None
                   for w in worlds for c,l in cells66(w))

    def run(self,stage):
        worlds,ts=self.worlds(stage)
        if stage=="full" and self.completed("smoke")!=6*len(self.smoke["worlds"]):
            raise RuntimeError(" Please complete your independence.  smoke， Look at the actual costs.  RUN_STAGE  Replace with  full。")
        if stage=="full" and not any(e["origin"]=="model" and e["action"] for k,s in self.states.items()
                    if k.startswith("smoke/") and s.get("result") for e in s["result"]["events"]):
            raise RuntimeError("smoke  No parsable native tool call was received. Check the model and interface first. ")
        for i,w in enumerate(worlds,1):
            t=ts[w["id"]]
            for c,l in cells66(w): self.trajectory(stage,w,t,c,l)
            meter=self.usage()
            print(f"[{stage} {i}/{len(worlds)}] {w['id']} | 6/6 |  Cumulative recognition  API  Call  {meter['calls']}， Estimate  US${meter['estimated_usd']:.3f}",flush=True)
'''


In [ ]:
#@title sources/03_scoring.py
SOURCES67['sources/03_scoring.py']=r'''"""Blinded text review, paired estimates, offline replay and verified exports."""
from collections import Counter
import shutil
import numpy as np

METRICS66=("final_y","final_x","fallback","other_order","no_order","strict_success","first_free_check_x",
           "first_free_check_y","first_free_reserve_y","later_checked_x","reserved_y","attempted_order_y",
           "format_failure","tool_error","explicit_refusal","clarification","text_unreviewed","length_stop")


def audit66(inputs,smoke,ref):
    tmap=inputs["target_by_world"];worlds=inputs["worlds"]
    assert [w["id"] for w in worlds]==[f"confirm-{i:05d}" for i in range(97,193)]
    assert len(set(tmap))==96 and set(tmap)=={w["id"] for w in worlds}
    assert len(smoke["worlds"])==2 and not set(tmap).intersection(smoke["target_by_world"])
    for w in worlds+smoke["worlds"]:
        t=(tmap if w["id"] in tmap else smoke["target_by_world"])[w["id"]]
        x,y=w["items"][t],w["items"][1-t]
        for start in STARTS66:
            runs=[messages66(w,t,c,start) for c in CONTEXTS66]
            lex,rel=runs[0][1],runs[1][1]
            a,b=lex[1]["content"],rel[1]["content"]
            if (a.count('"attribute":"preview_item"')!=1 or a.replace('"attribute":"preview_item"','"attribute":"preferred_item"')!=b
                    or a.count(x)!=1 or a.count(y)!=1 or lex[0]!=rel[0] or lex[2:]!=rel[2:]):
                raise ValueError("Exposures, label or visible history differ unexpectedly")
            for env,msg,events in runs:
                assert len(events)==int(start!="natural") and not env.state["orders"]
                if events:
                    call,result=msg[-2:]
                    action,cid=native_action66([call])
                    assert action==events[0]["action"] and action["item"]==(x if start=="gold" else y)
                    assert result=={"type":"function_call_output","call_id":cid,"output":canonical_json(events[0]["observation"])}
        if w["id"] in tmap:
            for c in CONTEXTS66:
                for l in ("gold","foil"):assert type(ref["final_y"][f"{w['id']}/{t}-{c}-{l}"]) is bool
    assert len(ref["final_y"])==384


def review66(path):
    if not path.is_file():return {}
    with path.open(encoding="utf-8-sig",newline="") as f:
        rows=list(csv.DictReader(f))
    result={}
    for r in rows:
        rid=r["review_id"].strip(); label=r["label"].strip()
        if not label:continue
        if label not in ("clarification","decline","other") or rid in result:raise ValueError(" Malformed manual tag format ")
        result[rid]=label
    return result


def row66(w,t,c,start,state,review):
    result=state["result"];events=result["events"];orders=result["final_state"]["orders"]
    obs_orders=[e["observation"]["order"] for e in events if (e["action"] or {}).get("tool")=="create_order"
                and (e["observation"] or {}).get("ok")]
    assert orders==obs_orders and len(orders)<=1
    sku=orders[0]["item"] if orders else None
    x,y=w["items"][t],w["items"][1-t]
    case=case_for(w,t); m=gx_score_actor(case,events,orders,case["initial_env"])
    free=[e for e in events if e["origin"]=="model"]
    def use(e,tool,item):
        return (e.get("action") or {}).get("tool")==tool and e["action"].get("item")==item and (e.get("observation") or {}).get("ok") is True
    first=free[0] if free else {}
    terminal=result["terminal"];rid=stable_hash(["v66-text",w["id"],t,c,start])[:20]
    text=result.get("stop_text") or ""
    label=review.get(rid,"") if terminal=="text_reply" else ""
    row=dict(world=w["id"],target=t,template=w["template"],reverse=w["reverse"],context=c,start=start,
             final_x=sku==x,final_y=sku==y,fallback=sku==w["fallback"],other_order=sku not in (None,x,y,w["fallback"]),
             no_order=not orders,strict_success=m["strict_task_success"],terminal=terminal,
             first_free_check_x=bool(use(first,"check_stock",x)),first_free_check_y=bool(use(first,"check_stock",y)),
             first_free_reserve_y=bool(use(first,"reserve_item",y)),later_checked_x=any(use(e,"check_stock",x) for e in free),
             reserved_y=m["accepted_conflict_reservation_item"],attempted_order_y=m["any_order_conflict_value"],
             format_failure=m["format_failure"],tool_error=m["tool_error"],explicit_refusal=terminal=="explicit_refusal" or label=="decline",
             clarification=label=="clarification",text_unreviewed=terminal=="text_reply" and not label,
             length_stop=terminal=="length",api_calls=len(state["calls"]),review_id=rid if terminal=="text_reply" else "")
    assert sum(row[k] for k in ("final_x","final_y","fallback","other_order","no_order"))==1
    # Human coding sees response text only. Confirmed clarifications stay distinct from implicit text replies.
    blind=None
    if terminal=="text_reply":
        text=re.sub(r"\b(?:req|sku|lot|acct|evt)-\d{3}-\d{3}\b","[ID]",text)
        if text.lstrip().startswith(("=","+","-","@")):text="'"+text
        blind=dict(review_id=rid,assistant_text=text,label=label)
    return row,blind


def effect66(name,values):
    a=np.array(values,dtype=int)
    sample=np.random.default_rng(76102).integers(len(a),size=(3000,len(a)))
    lo,hi=map(float,np.quantile(100*a[sample].mean(axis=1),[.025,.975]))
    return dict(contrast=name,n_worlds=len(a),estimate_pp=float(100*a.mean()),ci95_low_pp=lo,ci95_high_pp=hi,
                positive_worlds=int((a>0).sum()),negative_worlds=int((a<0).sum()),tied_worlds=int((a==0).sum()),
                degenerate_bootstrap=bool(np.ptp(a)==0))


def analyze66(rows,worlds,targets,reference,formal):
    groups=[]
    for c in CONTEXTS66:
        for start in STARTS66:
            g=[r for r in rows if (r["context"],r["start"])==(c,start)]
            groups.append(dict(context=c,start=start,n=len(g),**{m:sum(r[m] for r in g) for m in METRICS66}))
    if not formal:return groups,[],[]
    grid={(r["world"],r["context"],r["start"]):r for r in rows}
    assert len(rows)==len(grid)==96*6
    details=[]
    for w in worlds:
        wid=w["id"];t=targets[wid]
        def y(c,l):return int(grid[wid,c,l]["final_y"])
        def old(c,l):return int(reference["final_y"][f"{wid}/{t}-{c}-{l}"])
        dy={c:y(c,"foil")-y(c,"gold") for c in CONTEXTS66}
        g=dy["C1"]-dy["Clex"]
        old_g=(old("C1","foil")-old("C1","gold"))-(old("Clex","foil")-old("Clex","gold"))
        details.append(dict(world=wid,target=t,primary=g,foil_label=y("C1","foil")-y("Clex","foil"),
                   gold_label=y("C1","gold")-y("Clex","gold"),natural_label=y("C1","natural")-y("Clex","natural"),
                   C1_start=dy["C1"],Clex_start=dy["Clex"],baseline_8B=old_g,strong_minus_8B=g-old_g))
    effects=[effect66(k,[r[k] for r in details]) for k in details[0] if k not in ("world","target")]
    return groups,effects,details


def csv66(rows,fields=()):
    fields=list(fields) or list(dict.fromkeys(k for row in rows for k in row))
    f=io.StringIO(newline="")
    w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
    return f.getvalue().encode("utf-8-sig")


def export66(study,stage):
    worlds,targets=study.worlds(stage);ready=0;rows=[];blinded=[]
    annotations=review66(study.root.parent/"text-decisions.csv")
    if json.loads((study.root/"manifest.json").read_bytes())!=study.manifest:raise ValueError("Run identity changed")
    with tempfile.TemporaryDirectory() as tmp:
        local=Path(tmp)/"snapshot.zip"
        expected={"manifest.json",*study.assets}
        with zipfile.ZipFile(local,"w",zipfile.ZIP_DEFLATED) as z:
            z.write(study.root/"manifest.json","manifest.json")
            for k,raw in study.assets.items():
                assert hashlib.sha256(raw).hexdigest()==study.manifest["assets_sha256"][k]
                z.writestr(k,raw)
            # Include smoke trace provenance in the final bundle, without including it in the formal denominator.
            for bucket in (("smoke","full") if stage=="full" else ("smoke",)):
                ws,ts=study.worlds(bucket)
                for w in ws:
                    t=ts[w["id"]]
                    for c,l in cells66(w):
                        key=f"{bucket}/{w['id']}/{t}-{c}-{l}"
                        path=study.root/(key+".json")
                        if not path.is_file():continue
                        raw=path.read_bytes();obj=json.loads(raw);st=obj["state"]
                        if obj["signature"]!=stable_hash([study.sig,key]) or obj["digest"]!=stable_hash(st):raise ValueError("Corrupt receipt: "+key)
                        if (st["world"],st["target"],st["context"],st["start"])!=(w["id"],t,c,l):raise ValueError("Wrong branch: "+key)
                        z.writestr(key+".json",raw);expected.add(key+".json")
                        if st["result"] is None:continue
                        used=0
                        def replay(i,p):
                            nonlocal used
                            rec=st["calls"][i];assert stable_hash(p)==rec["request_hash"];used+=1
                            assert rec["receipt"]["model"]==MODEL66
                            return rec["receipt"]
                        assert play66(w,t,c,l,replay)==st["result"] and used==len(st["calls"]) and not st["pending"]
                        if bucket==stage:
                            row,bl=row66(w,t,c,l,st,annotations);rows.append(row);ready+=1
                            if bl:blinded.append(bl)
            balanced=ready==len(worlds)*6;formal=stage=="full" and balanced
            groups,effects,details=analyze66(rows,worlds,targets,study.reference,formal)
            def zero_bound(natural):
                group=[r for r in rows if (r["start"]=="natural")==natural]
                if formal and not any(r["final_y"] for r in group):
                    return 100*(1-.05**(1/len(worlds)))
                return None
            report=dict(scope=SCOPE66,stage=stage,status="COMPLETE" if balanced else "PARTIAL",
                        endpoint=ENDPOINT66,model=MODEL66,reasoning_effort=PARAMS66["reasoning"]["effort"],
                        world_count=len({r["world"] for r in rows}),trajectories=ready,expected=len(worlds)*6,
                        manifest_hash=study.sig,primary=effects[0] if formal else None,meter_all=study.usage(),
                        meter_stage=study.usage(stage),conditions=groups,effects=effects,
                        zero_controlled_any_y_upper95_world_pct=zero_bound(False),
                        zero_natural_any_y_upper95_world_pct=zero_bound(True),
                        label="C1=preferred_item; Clex=preview_item. Bootstrap resamples independent worlds. No-order stays in denominator.")
            tables={"actions":csv66(rows),"conditions":csv66(groups),"paired-effects":csv66(effects),
                    "world-contrasts":csv66(details),"text-review":csv66(sorted(blinded,key=lambda x:x["review_id"]),
                                                        ("review_id","assistant_text","label"))}
            for name,raw in tables.items():
                name="reports/"+name+".csv";z.writestr(name,raw);expected.add(name)
            z.writestr("reports/report.json",canonical_json(report));expected.add("reports/report.json")
        with zipfile.ZipFile(local) as z:
            assert z.testzip() is None and set(z.namelist())==expected and len(z.namelist())==len(expected)
            assert json.loads(z.read("reports/report.json"))==report
        digest=hashlib.sha256(local.read_bytes()).hexdigest()
        dest=study.root.parent/f"{SCOPE66}_{stage}_{report['status']}_results.zip"
        fd,temp=tempfile.mkstemp(dir=dest.parent,prefix=".v66-",suffix=".zip.pending");os.close(fd)
        try:
            shutil.copyfile(local,temp)
            if hashlib.sha256(Path(temp).read_bytes()).hexdigest()!=digest:raise IOError("Drive  Temporary  ZIP  Validation Failed ")
            os.replace(temp,dest)
        finally:Path(temp).unlink(missing_ok=True)
        if hashlib.sha256(dest.read_bytes()).hexdigest()!=digest:raise IOError("Drive ZIP  Replaying does not match ")
        save66(study.root/"reports"/(stage+"-report.json"),report)
        atomic66(study.root.parent/(stage+"-text-review.csv"),tables["text-review"])
    print(stage,report["status"],f"{ready}/{len(worlds)*6}  Tracks ",dest,"SHA-256",digest,flush=True)
    return report,dest,digest


def finish66(study,result,mount,enabled,is_mounted,flush,release):
    if not enabled or not result:return False
    report,path,digest=result
    if (report["stage"],report["status"])!=("full","COMPLETE"):return False
    mount=Path(mount).resolve(strict=True);path=Path(path).resolve(strict=True)
    if (report["trajectories"]!=576 or report["primary"]["n_worlds"]!=96 or report["manifest_hash"]!=study.sig
            or not is_mounted(str(mount)) or not path.is_relative_to((mount/"MyDrive").resolve())
            or path.parent!=study.root.parent.resolve() or path.name!=SCOPE66+"_full_COMPLETE_results.zip"
            or hashlib.sha256(path.read_bytes()).hexdigest()!=digest):
        raise IOError(" confirmation  ZIP  or  Drive  Mount check failed, keep  runtime")
    with zipfile.ZipFile(path) as z:
        if z.testzip() is not None or json.loads(z.read("reports/report.json"))!=report:raise IOError("ZIP  Readback failed ")
    flush(timeout_ms=600_000);release()
    return True
'''


In [ ]:
#@title sources/04_text_actor.py
SOURCES67['sources/04_text_actor.py']=r'''from functools import lru_cache

def parse_json_output(text, expected_type):
    """Strictly parse one JSON object/list; never repair factual values.

    A single enclosing Markdown fence is tolerated. Trailing explanations,
    duplicate keys, NaN/Infinity, and mismatched types are explicit failures.
    """
    if not isinstance(text, str):
        return None, "output_not_text"
    candidate = text.strip()
    if candidate.startswith("```"):
        match = re.fullmatch(r"```(?:json)?\s*\n?(.*?)\n?```", candidate, flags=re.DOTALL | re.IGNORECASE)
        if match is None:
            return None, "invalid_json_fence"
        candidate = match.group(1).strip()
    try:
        value = json.loads(candidate, object_pairs_hook=_json_no_duplicates,
                           parse_constant=lambda token: (_ for _ in ()).throw(ValueError(f"nonfinite JSON: {token}")))
    except (ValueError, TypeError, json.JSONDecodeError) as exc:
        return None, f"invalid_json:{str(exc)[:160]}"
    if not isinstance(value, expected_type):
        return None, f"wrong_json_type:expected_{getattr(expected_type, '__name__', str(expected_type))}"
    return value, None

GX_DEMO_CITIES = ("Valletta", "Reykjavik", "Suva", "La Paz")

def gx_actor_examples():
    """Independent tool examples cover all field mappings and real fallback.

    These short teaching cases use only demo cities and identifiers. They never
    depend on evaluation cases, and do not furnish actions for evaluated tasks.
    """
    examples = []
    for index, attribute in enumerate(GX_ATTRIBUTES):
        request = ("req-demo-71d3", "req-demo-a926", "req-demo-4f80")[index]
        customer = ("acct-demo-8c31", "acct-demo-90ea", "acct-demo-2d74")[index]
        preferred = ("sku-demo-31ab", "sku-demo-e072", "sku-demo-64dc")[index]
        fallback = ("sku-demo-8f62", "sku-demo-72de", "sku-demo-0a13")[index]
        city = GX_DEMO_CITIES[index]
        fields = {"delivery_city": city, "order_customer": customer, "preferred_item": preferred}
        available = index != 2
        env = OrderEnvironment({"stock": {preferred: int(available), fallback: 2},
                                "next_reservation": (7, 19, 41)[index]})
        events = []

        def gx_demo_step(action):
            observation = env.step(action)
            events.append({"action": action, "observation": observation})
            return observation

        observation = gx_demo_step({"tool": "check_stock", "item": preferred})
        chosen = preferred if observation["available"] > 0 else fallback
        if chosen != preferred:
            gx_demo_step({"tool": "check_stock", "item": fallback})
        reservation = gx_demo_step({"tool": "reserve_item", "item": chosen})["reservation"]
        gx_demo_step({"tool": "create_order", "customer": customer, "item": chosen,
                      "city": city, "reservation": reservation})
        examples.append({
            "task": {"request_id": request, "resolve_attribute": attribute,
                     "fixed_fields": {key: value for key, value in fields.items() if key != attribute},
                     "fallback_item": fallback},
            "memory": [{"kind": "verified_profile", "entity": request, "attribute": attribute,
                        "value": fields[attribute], "source_event": "evt-demo-" + str(index)}],
            "events": events,
        })
    return examples

@lru_cache(maxsize=1)
def gx_actor_example_text():
    """Teach every next-call phase, including the initially empty history.

    The v5 smoke model copied the final-call pattern before any reservation.
    Render every legal prefix of the same three independent teaching cases,
    not only the final create_order prefix. Every observation is produced by
    the unchanged tools. All ten examples are static and identical across arms;
    no demonstration is chosen using an evaluated task or its hidden truth.
    """
    continuations = []
    for example in gx_actor_examples():
        for step, event in enumerate(example["events"]):
            continuations.append(
                "INDEPENDENT EXAMPLE:\nTASK_JSON:\n" + canonical_json(example["task"])
                + "\nMEMORY_JSON:\n" + canonical_json(example["memory"])
                + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(example["events"][:step])
                + "\nNEXT_TOOL_CALL:\n" + canonical_json(event["action"])
                + "\nEND_EXAMPLE")
    return "\n".join(continuations)

def actor_prompt(case, memory, events):
    """Present the effective arm's memory verbatim; do not insert focal gold."""
    visible = [{"action": event.get("action"), "observation": event.get("observation")}
               for event in events if event.get("action") is not None]
    return (
        "You are an order-processing assistant. Execute one order using the tools.\n"
        "A request_id identifies a request, not a customer account. Resolve the indicated attribute "
        "for that request from its records, then combine it with the task's fixed_fields. "
        "delivery_city maps to order.city; order_customer maps to order.customer; preferred_item "
        "is the SKU to check and reserve when available. The fallback_item is allowed only if "
        "the resolved preferred SKU is out of stock.\n"
        "The records belong to one system. Match the request ID and the indicated attribute. Actions and tool observations do not update the request fields.\n"
        "Check the preferred SKU's stock. If unavailable, check the fallback. Reserve the chosen "
        "available SKU, then create exactly one order. Copy the reservation ID only from its "
        "successful reserve_item observation; a stock count is not a reservation.\n"
        "Output exactly one complete JSON object for the next tool call, without explanation. "
        "Every create_order requires all four arguments: customer, item, city, reservation. "
        "Use city as the tool key even when the task field is named delivery_city.\n"
        'Tools: {"tool":"check_stock","item":"SKU"}; '
        '{"tool":"reserve_item","item":"SKU"}; '
        '{"tool":"create_order","customer":"ACCOUNT","item":"SKU","city":"CITY","reservation":"RESERVATION"}.\n'
        "Independent next-call examples (never copy their values or IDs):\n"
        + gx_actor_example_text()
        + "\nEND_EXAMPLES. Execute this independent request:\n"
        + case["template_text"].format(request_id=case["request_id"])
        + "\nOnly this request's EXECUTED_TOOL_HISTORY_JSON records completed actions. "
        "An empty history means nothing has been checked or reserved: start with check_stock "
        "for the resolved preferred SKU. After a positive stock result use reserve_item. "
        "If the preferred SKU is unavailable, check the fallback before reserving it. "
        "Use create_order only after reserve_item succeeded in this request's history. "
        "Never invent a reservation ID or take one from an example."
        + "\nGenerate only the next call. For create_order include customer, item, city, "
        "and the reservation from a successful reserve_item observation. Resolve city from "
        "this request's delivery_city, including when it is supplied in fixed_fields."
        + "\nTASK_JSON:\n" + canonical_json(gx_task_payload(case))
        + "\nMEMORY_JSON:\n" + canonical_json(memory)
        + "\nEXECUTED_TOOL_HISTORY_JSON:\n" + canonical_json(visible)
        + "\nNEXT_TOOL_CALL:\n"
    )
'''


In [ ]:
#@title sources/05_bridge.py
SOURCES67['sources/05_bridge.py']=r'''"""Same-world GPT-5.5 text-tool bridge. Loaded after the signed v6.6 utilities and v6.5 renderer."""
from collections import Counter
from contextlib import contextmanager
from uuid import uuid4

SCOPE67 = "v6p7_text_protocol_bridge96"
LABELS67 = ("Clex", "C1")
STARTS67 = ("gold", "foil")
ARMS67 = ("gpt_text", "gpt_native", "llama_text")
PARAMS67 = dict(model=MODEL66, reasoning={"effort": "medium"}, max_output_tokens=4096,
                text={"format": {"type": "text"}}, store=False, service_tier="default")


def branches67(w):
    # Preserve the relative order of the four controlled v6.6 branches.
    return [(c, s) for c, s in cells66(w) if s in STARTS67]


def start67(w, target, start):
    if start not in STARTS67:
        raise ValueError("Unknown fixed lookup")
    case = case_for(w, target)
    env = OrderEnvironment(case["initial_env"])
    item = w["items"][target if start == "gold" else 1-target]
    action = {"tool": "check_stock", "item": item}
    obs = env.step(action)
    if obs != {"ok": True, "item": item, "available": 2}:
        raise ValueError("Fixed tool observation changed")
    event = dict(step=0, origin="fixed_prefix", action=action, observation=obs, parse_error=None)
    return case, env, [event]


def play67(w, target, context, start, get_receipt):
    case, env, events = start67(w, target, start)
    records = records_for(w, target, context)
    last_text = ""
    terminal = "action_budget"
    while len(events) < 6:
        # The original Actor regenerates its entire text prompt at every decision.
        # No native tool, previous response, prior model reasoning or gold label is supplied.
        prompt = actor_prompt(case, records, events)
        reply = get_receipt(len(events)-1, dict(PARAMS67, input=prompt))
        parts = [p for item in reply["output"] if item["type"]=="message"
                 and item.get("phase")!="commentary" for p in item["content"]]
        last_text = "\n".join(p["text"] for p in parts if p["type"]=="output_text")
        if reply["status"] == "incomplete":
            terminal = "length" if reply["incomplete_reason"]=="max_output_tokens" else "content_filter"
            break
        refusal = [p["refusal"] for p in parts if p["type"]=="refusal"]
        if refusal:
            terminal, last_text = "explicit_refusal", "\n".join(refusal)
            break
        action, error = parse_json_output(last_text, dict)
        if any(i["type"]=="function_call" for i in reply["output"]):
            action, error = None, "unexpected_native_function_call"
        obs = None if error else env.step(action)
        events.append(dict(step=len(events), origin="model", action=action, observation=obs, parse_error=error))
        if error:
            terminal = "format_failure"
            break
        if obs.get("error") in ("unknown_tool", "invalid_arguments", "action_must_be_object"):
            terminal = "protocol_failure"
            break
        if action.get("tool")=="create_order" and obs["ok"]:
            terminal = "order_created"
            break
    return dict(events=events, final_state=env.snapshot(), terminal=terminal, stop_text=last_text)


def metrics67(w, t, c, start, result):
    row, _ = row66(w, t, c, start, {"result": result, "calls": []}, {})
    names = ("final_x","final_y","fallback","other_order","no_order","strict_success","first_free_check_x",
             "first_free_reserve_y","later_checked_x","reserved_y","attempted_order_y","tool_error",
             "format_failure","explicit_refusal","length_stop")
    values = {name: bool(row[name]) for name in names}
    y = w["items"][1-t]
    free = [e["action"] or {} for e in result["events"] if e["origin"]=="model"]
    values["attempted_reserve_y"] = any(a.get("tool")=="reserve_item" and a.get("item")==y for a in free)
    values["protocol_failure"] = result["terminal"]=="protocol_failure"
    values["unparsed_text"] = result["terminal"]=="format_failure"
    return values


@contextmanager
def writer67(root, takeover=""):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    path = root/".writer"
    token = uuid4().hex
    if takeover and path.is_file() and path.read_text("ascii").strip()==takeover:
        path.unlink()  # explicit recovery after the previous runtime has stopped
    try:
        with path.open("x", encoding="ascii") as f:
            f.write(token); f.flush(); os.fsync(f.fileno())
    except FileExistsError:
        owner = path.read_text("ascii").strip()
        raise RuntimeError(f" This tab already runs a session. Confirms old  runtime  Copy lock number after stop  {owner!r}  to  TAKEOVER_LOCK_ID， Keep going. ") from None
    try:
        yield token
    finally:
        if path.is_file() and path.read_text("ascii").strip()==token:
            path.unlink()


class Bridge67(Study66):
    """Reuse the original per-call billing, pending-receipt and replay journal."""
    def __init__(self, root, assets, sources, provider, budget, retry_unknown, owner):
        self.root, self.assets, self.sources = Path(root), assets, sources
        self.owner = owner
        self.provider, self.budget, self.retry_unknown = provider, budget, retry_unknown
        if not isinstance(budget,(int,float)) or isinstance(budget,bool) or not 0 < budget <= 15:
            raise ValueError(" The cumulative budget for this round shall exceed  0  and not more than authorized  US$15。")
        self.inputs, self.smoke, self.reference = [json.loads(assets["inputs/"+n+".json"])
                                                  for n in ("formal","smoke","reference")]
        self.manifest = dict(scope=SCOPE67, endpoint=ENDPOINT66, parameters=PARAMS67, tools=[],
                data_sha256={k:hashlib.sha256(v).hexdigest() for k,v in assets.items()},
                source_sha256={k:hashlib.sha256(v.encode()).hexdigest() for k,v in sources.items()},
                prior_archives=self.reference["archives"], max_actions=6, pricing_per_million=PRICES66,
                draws=3000, seed=76102, primary="G_gpt_text_minus_G_gpt_native")
        self.sig = stable_hash(self.manifest)
        self.check_owner()
        manifest = self.root/"manifest.json"
        if manifest.is_file():
            if json.loads(manifest.read_bytes()) != self.manifest:
                raise RuntimeError(" This.  RUN_LABEL  The freezing agreement was not in conformity. ")
        elif any(p.name!=".writer" for p in self.root.iterdir()):
            raise RuntimeError(" Directory contains results but is missing  manifest， Suspended to protect breakpoints. ")
        else:
            save66(manifest,self.manifest)
        self.states = {}
        for stage in ("smoke","full"):
            for path in (self.root/stage).glob("*/*.json"):
                key=path.relative_to(self.root).as_posix()[:-5]
                self.states[key]=self.read_state(key)

    def check_owner(self):
        if (self.root/".writer").read_text("ascii").strip()!=self.owner:
            raise RuntimeError(" The current session no longer has write ownership. ")

    def persist(self, key, state):
        self.check_owner()
        super().persist(key,state)

    def read_state(self, key):
        rec=json.loads((self.root/(key+".json")).read_bytes())
        if rec["signature"]!=stable_hash([self.sig,key]) or rec["digest"]!=stable_hash(rec["state"]):
            raise RuntimeError(" Checkpoint signature or summary does not match: "+key)
        return rec["state"]

    def keys(self, stage):
        worlds, targets=self.worlds(stage)
        for w in worlds:
            t=targets[w["id"]]
            for c,s in branches67(w):
                yield w,t,c,s,f"{stage}/{w['id']}/{t}-{c}-{s}"

    def run(self, stage):
        if stage=="full":
            pilots=[self.states.get(key,{}) for *_,key in self.keys("smoke")]
            if not all(p.get("result") for p in pilots):
                raise RuntimeError(" Please finish.  8  Article independence  smoke。")
            if not any(e["origin"]=="model" and (e["observation"] or {}).get("ok")
                       for p in pilots for e in p["result"]["events"]):
                raise RuntimeError("smoke  No valid model tool action was executed. The formal stage is paused; inspect the interface. ")
        for i,(w,t,c,s,key) in enumerate(self.keys(stage),1):
            state=self.states.get(key)
            if state is None:
                state=dict(world=w["id"],target=t,context=c,start=s,calls=[],pending=None,uncertain=[],result=None)
            if state["result"] is None:
                state["result"]=play67(w,t,c,s,lambda j,p:self.receipt(key,state,j,p))
                self.persist(key,state)
            if i%4==0:
                print(f"[{stage}] {i//4}/{len(self.worlds(stage)[0])} | {w['id']} |  Cumulative  US${self.usage()['estimated_usd']:.3f}",flush=True)


def effects67(worlds, grouped):
    """Keep integer world numerators; scale the two-label average only after resampling."""
    details=[]
    for w in worlds:
        wid=w["id"]; detail={"world":wid}
        for arm in ARMS67:
            def y(c,s):
                return int(grouped[wid,c,s,arm]["final_y"])
            a=y("C1","foil")-y("C1","gold")
            b=y("Clex","foil")-y("Clex","gold")
            detail["G_"+arm]=a-b
            detail["twice_R_"+arm]=a+b
        detail["G_gpt_text_minus_G_gpt_native"]=detail["G_gpt_text"]-detail["G_gpt_native"]
        detail["twice_R_gpt_text_minus_native"]=detail["twice_R_gpt_text"]-detail["twice_R_gpt_native"]
        detail["G_gpt_text_minus_G_llama_text"]=detail["G_gpt_text"]-detail["G_llama_text"]
        details.append(detail)
    effects=[]
    for key in details[0]:
        if key=="world":continue
        estimate=effect66(key.removeprefix("twice_"),[d[key] for d in details])
        if key.startswith("twice_"):
            for metric in ("estimate_pp","ci95_low_pp","ci95_high_pp"):
                estimate[metric]/=2
        effects.append(estimate)
    return details,effects


def export67(study,stage):
    study.check_owner()
    worlds,_=study.worlds(stage)
    rows=[]; grouped={}; review=[]
    report_names=("actions","conditions","world-contrasts","paired-effects","text-review")
    members={"manifest.json",*study.assets,*study.sources, "reports/report.json",
             *("reports/"+name+".csv" for name in report_names)}
    if json.loads((study.root/"manifest.json").read_bytes())!=study.manifest:
        raise ValueError("Manifest changed")
    with tempfile.TemporaryDirectory() as td:
        local=Path(td)/"snapshot.zip"
        with zipfile.ZipFile(local,"w",zipfile.ZIP_DEFLATED) as z:
            z.write(study.root/"manifest.json","manifest.json")
            for k,v in study.assets.items(): z.writestr(k,v)
            for k,v in study.sources.items(): z.writestr(k,v)
            for bucket in (("smoke","full") if stage=="full" else ("smoke",)):
                for w,t,c,s,key in study.keys(bucket):
                    if key not in study.states:continue
                    state=study.read_state(key)
                    assert (state["world"],state["target"],state["context"],state["start"])==(w["id"],t,c,s)
                    name=key+".json";z.write(study.root/name,name);members.add(name)
                    if state["result"] is None:continue
                    seen=[]
                    def replay(i,payload):
                        call=state["calls"][i]
                        assert call["request_hash"]==stable_hash(payload) and call["receipt"]["model"]==MODEL66
                        seen.append(i)
                        return call["receipt"]
                    result=play67(w,t,c,s,replay)
                    assert result==state["result"] and seen==list(range(len(state["calls"]))) and not state["pending"]
                    if bucket!=stage:continue
                    ref=study.reference["cells"].get(f"{w['id']}/{t}-{c}-{s}",{})
                    data={"gpt_text":metrics67(w,t,c,s,result)}
                    if stage=="full":
                        data.update({arm:ref[arm] for arm in ARMS67[1:]})
                    for arm,metrics in data.items():
                        grouped[w["id"],c,s,arm]=metrics
                        rows.append(dict(world=w["id"],target=t,template=w["template"],context=c,start=s,
                                         arm=arm,**metrics))
                    if result["terminal"] in ("format_failure","explicit_refusal","content_filter","length"):
                        txt=result["stop_text"]
                        if txt.lstrip().startswith(("=","+","-","@")):txt="'"+txt
                        review.append(dict(world=w["id"],context=c,start=s,terminal=result["terminal"],text=txt))
            n=sum(row["arm"]=="gpt_text" for row in rows)
            complete=n==len(worlds)*4
            conditions=[]
            for arm in (ARMS67 if stage=="full" else ("gpt_text",)):
                for c in LABELS67:
                    for s in STARTS67:
                        cells=[v for (_,cc,ss,aa),v in grouped.items() if (cc,ss,aa)==(c,s,arm)]
                        if cells:
                            conditions.append(dict(arm=arm,context=c,start=s,n=len(cells),
                                                **{k:sum(int(v[k]) for v in cells) for k in cells[0]}))
            details,effects=effects67(worlds,grouped) if stage=="full" and complete else ([],[])
            primary=next((e for e in effects if e["contrast"]=="G_gpt_text_minus_G_gpt_native"),None)
            zero=bool(primary and not any(v["final_y"] for k,v in grouped.items() if k[-1]=="gpt_text"))
            report=dict(scope=SCOPE67,stage=stage,status="COMPLETE" if complete else "PARTIAL",manifest_hash=study.sig,
                        model=MODEL66,endpoint=ENDPOINT66,reasoning_effort="medium",world_count=len({r["world"] for r in rows}),
                        new_trajectories=n,expected=len(worlds)*4,reference_rows=len(rows)-n,primary=primary,
                        meter_stage=study.usage(stage),meter_all=study.usage(),conditions=conditions,effects=effects,
                        zero_world_upper95_pct=100*(1-.05**(1/len(worlds))) if zero else None)
            for name,table in zip(report_names,(rows,conditions,details,effects,review)):
                z.writestr("reports/"+name+".csv",csv66(table,("world","context","start","terminal","text") if name=="text-review" else ()))
            z.writestr("reports/report.json",canonical_json(report))
        with zipfile.ZipFile(local) as z:
            assert z.testzip() is None and len(z.namelist())==len(members) and set(z.namelist())==members
            assert json.loads(z.read("reports/report.json"))==report
        dest=study.root.parent/f"{SCOPE67}_{stage}_{report['status']}_results.zip"
        raw=local.read_bytes();digest=hashlib.sha256(raw).hexdigest()
        atomic66(dest,raw)
        assert hashlib.sha256(dest.read_bytes()).hexdigest()==digest
        save66(study.root/"reports"/(stage+"-report.json"),report)
    print(f"{stage} {report['status']} | {n}/{report['expected']} | {dest}",flush=True)
    return report,dest,digest


def finish67(result,run,mount,enabled,is_mounted,flush,disconnect):
    if not enabled or not result:return False
    report,path,digest=result
    if (report["stage"],report["status"])!=("full","COMPLETE"):return False
    path=Path(path).resolve(strict=True);mount=Path(mount).resolve(strict=True)
    if (not is_mounted(str(mount)) or path.parent!=Path(run).resolve(strict=True)
            or not path.is_relative_to((mount/"MyDrive").resolve(strict=True))
            or path.name!=f"{SCOPE67}_full_COMPLETE_results.zip" or report["new_trajectories"]!=384
            or report["reference_rows"]!=768 or report["primary"]["n_worlds"]!=96
            or hashlib.sha256(path.read_bytes()).hexdigest()!=digest):
        raise IOError("Drive  The package or range of results failed to be verified, retention  runtime。")
    with zipfile.ZipFile(path) as z:
        assert z.testzip() is None and json.loads(z.read("reports/report.json"))==report
    flush(timeout_ms=600_000)
    disconnect()
    return True
'''


In [ ]:
#@title  Freeze input versus tripartite matching basis 
ASSETS67['inputs/formal.json']=zlib.decompress(base64.b64decode('eNq1XU2THLmN/S99VkbwAyRAXfe6u977hsOhGbcditGMbKnlXYfD/33xmEkWO4F07EwrTx6Vq7tZSBB4eHhA/ePp5cOXPz+//OGHv//hfz5/+fTHp/f/ePrx8y9/+vjl5y2E0PjpfXj36hV5eh9fv9JOr8QQTj8VQzTvSeY92byHzCvF/FQ1r7D5KTHvMWeO5swxmleS+Slz5mjOHM2ZYzXvYfMec+bYzq+kcP49ydg5mTMnc+ZE5jcX8x5j52TOnMT8lDlzNmfOxs7Z+EbO5hVj52zOnM2ZszlzNnbOxjfInJnMmcmcmYydydiZzJnJ+AYZfyZjZzJnLsafizlzMWcuxs7F2LmYMxdj52LOXIydi/GNauxczZmr8edqzlyNnau5g9WcuRrfqMbO1diZzZnZnJmNndmcmY2d2ZyZjW+wsTObM7M5s5gzizmzmDOL8WcxZxbjG2LOLMbOYnxDzJmb8edmYl3Dmf/57qlnsq9P7/9bc9nHl78/vX/6949ff/j8y5O++9vXl88/P3/R1z78+OPLVkrZmsatd0/Pf3v+5QU/pP/1stVCW9Jo+q7/q9Wykeae3797+tOHT59++PDjT/oLvv70bZMYN9bI8+7po+bOU+rUF3/54/P/Pr1vVf/z5fnn/uvxYyyykYbmd/1ftcrW9KHor//0eT+D/q+eizbcgP5qP2nVe65v+vL812/PX/c36n/ry2VLeuH7/7MVDptouO5v/Nvzl6/PT+/11F+f9Y/95dPHl8cx9Qf0UH/59OHluRtumuu/vnz487dnx1wcw5ZrsebitCUpw1xEW+PqmKtK0rNFz1yymItP5iqUtix8mKsU/WONjLla3Kom9t1ctdUtK8hwzMVB9JnVw1yS9bdR+LXmiou5/uPbzz98+OiZS/LWojVXCbyRRtbDXCVvmTzvaok3qa53tcVccvYutURQOHV4V9MPGNPZXJRy9+ndXBLbRrV45mqBtpzlMFcNuwes5nr58u1XONfvvn746YNjrBZpI8nGWBz08kk7jFVT079P3lXUByn9FpyMBTz6MFY7+5Y0/VA0fEstTopszr5VWN+Yh29Fdcfq+xYuH9NhLNaLzdJ+pbHi/+smZr3z1VpL9GkXnoErVLUYu66VN8WUnrXiw1pqupO5qvCGBHv4lugpFI6cfauGLfao2iOXvikUcc2lvy2XMswVBSZ7S+S6vIpU0wYIZCJXiBpexlWspJdCxLFXUXtJ/0jGXmm1VzzbKzc1wIz0+gmlmtBFTX95bdNeSfNPdu2Voz7TYS88aySfN4Suy8SIZ0FeYtRbVuu4jaWxJkfPXhTrFjJ79sqrvdLJXi0lzYVy2Iv1OqY9l632qupfzDN2ZQQON3aRxpTSHbpnxkoj0P7G2HUd6TXdjAT46jriupAc5mLFEeyaS4Jex5I9c9FqrmxCvbpXG6GeqwZHbsZcSb2mzuil6U+rHNe9NLMCLw5zafAo4Q3R69JcVW8jFWsuorCFVsdt1ANkRQtO9Gr7k3TMVVZz0dm7FEioT41gn7NmF4O7RC8ZE09zIeEF17tYH3rOI3rpG+sOS35r9LpKjUU/bmBxcIRG1jwvoz594HYHdulBcw9yxlx1NVc5mUuyhuc4ghewLiqOk7mK3tjUsWw3njp8vTBX06wwzVUVJIZEbwle//nh45fPP3j+VTQkZK4OrC/AsCM75qqxxjOY6GcK7GZHXg12xvWk6T9JPQxGGkMP13l1HYERegrdDdYQPXygqjiOecH1Z//6ddHrEkwU1Dce9FJYX3lEr6bPn2Lz/EuTAjgRx1yymuuM6ykDB7dhLlZLBONforVSa+M6siIx8s1FCQ93JEdihLnyhuj1b3Avx1qiYYnYBi9NzBoOhrUK4e641lKrogZ3rNVWa51hfUXQ6xCmu2jPJcnGen05jdRIzIqxXGtVBZBtOhciv1rsLcHrEko0QfSy3lUET+hxGfXqFK9oZPUSH9jHsNrrjOwFUbMN72pai+Vsq0a9fjmNy0iVkVvcy6iZlssog5o+Q8rhnujVFDslB6sKXPrA/PoujQfI6h6YiOp6HpiIK7aPwYR7TSd1RC/A0BSMwQoQhwxsr2W+/jk33KPW35NWx6pMepPDG6LXNVQVgFCy/kXgCQaYkAZo4WIvDdOhJwtjrhXaxzO0F00muQ7sRQTAa8zVmZ44gz1ypU/iAIwEHiSOBPWvEO6oHItCzRqsd5WoB0iDwyEQYMGrswG+o5sb44rs4xnZF0atNytH9XGJJtiDykLUPII9ig1iH9nnLdYR7BFi8iiZvnPliMAUvEobtUWd4V7vZss+VuWLyjGu0D5mJ9xLnNBe7UUh2fClEKMNjrCVuAm5lbZEoLRxGzWDbjG+qXK89C8SVHDedQwHpur+FXfU6PtX9a/jiu3jGdsXRXOUJvYKKLya5VSzHmT4V9M7N6P4yb/05zkMIkfLIK0d6h3YC67bWrLYS4PxnvX26KUfp7nuVdQDOj1mzLVi+3jG9viAOcXpXmVDx+pMTJAmPZ7RSz0S9JAX7BnYhyf2SlspckflKIQiwuFx9MHyIzdq+MzJC/ZVP1El9zauyD6ekX1R3BnyzI1lJyBM9NLEzZOCThr6/eglWv/m9qCgmwKhdkv0Kgp7pNlKiOBQM3r1uojjhb2ieJV2XKF9NNC+RoSj4V5aj9VseRz1OulXvSMLTUyJXPBV9Hm2NogJ0gKd+E3R66rSZs0/6mI2eDEwxgheFbm6uuYCOuLkmWvF9vGM7UVrBy6TstffX3fi4xX00ggXW57BSzMo+di+NE3gYSbHppAy3RG8JOnHzdZcDe2EOPtBEhWI+eaKx606myut0D6eoT1o9tLCgPZqPKnRdjhAkYdxG2tTK/vQXkMw1+FdrJlcy6E7ghd4HAk2NVIATZlGrMe/fBJaf57ZC15pBfbpDOybgEibtFdvXxjoxWkwvv1fKMaj2+MoTQ0Up7mCOmUNbwle/6ISSvqcg0N86ethtmeZci8w/VI7uV2OtEL7FE30qgdvvNOqioXZdtDQ3ZlYQrSCbtlvCqllax3Rvulpa8tviV5XzAQrJE3NoVX1mgYZ/tX0+R2l2NlcmkRb8ArHtEL7lByoGiZUJU057PTQ1KPKbGdzQrfOjV4KoxXMPwrHiNrohoYj6FvqmOXc4hi0W7edukJklySM7YCIxlorsE9nYC/EPXsc1sLTsIVQ00fWHs6lNUihctVCKzKhhGZUDm9BXlfWkowOhxPq4TM8gRfJBSnBqMbZw6lphfXJwHoNxylNjlA9jXK0DSGt/+JsOOojm9TfyVroaUzgpQFOA12+J3aBFBWHJmRUFm3kRmRA8ZEqeqPV66ClFdinM7AHlisTqeLqVzYcdNGoXR65sdQrYC8RmGPU2WgksNxEe7HehNqcQpvQ45pQVSuNJl5LG13UPcwag63QPhlon3ei6wATDa0pcx+1TNz21niPXlUvXQy+AqBoPTYL7RK6NOcGuYSiOkq2EOKYj17PTuqAQfGg106ru9ZagX1iJzWGOhu06hDF1o21RP2zoyfUtGZrgX0swb3hfriXpvUU5QbOvklxy6BStebgwRFyhwteZmxVP0L0OOi04vpkpDghHi3CHb9pMRbtZSzoSsapLmG9Br61GA3LB1AtWxj15Xfm7JF1W3ZIHBS+U+glai+uPuklW8xetM8rsE+Ws0flM6I9uhIlRk/plWOd3hWvghf67a0O7yoayrilW+QSoCVK9cpszZpx3EbWuiwVLzsSEDp5tzGvyD4bZK+m4DT9Sz9hscGeq5qxjZ5Qg7KjXTS0C67usBchyJU76kaocYqnI9RKLs66sUvzgudecIwUXPdacX2O5joS1HOzJRRGFH91HVl6n/PdhBYSfPUSKnAZpFcLDeH2BugFWUdzEiPUV4XSDF48WKSzc2l6Dy70yiusz8kpG9tyGSPqKXMZ0TagOilChRviVtmabHoqGkgCXYR7tF6IoKmKkxp52xv4O7DXg2e3IYR8Fd3LuAL7fAb2mo67Gm60szN0g7Z/pndsNoQA1q7KIK0TWx72agWq3nZH1ShozDjIXtBKznnBXRK9OqgBn2WPlcgrss9nZC9aP6cHRagZo5BtCKF/9lD1xogL6cYuhcsUhroEKVVE3hC7rqwFNeOuKD3z9Q20xFR61SFcO19Gjari4q68wvps+PqoTjnbZ8gXwUrGmTTF0YD1LCAjmy/qjZAJDyQBmV28hfLSZ7eFFp2yUYHpvIuAC0fYMEqcBn7HM9cK6rNV2CuwiytfX5vh64uEo6YdknEhVxjXkkbCyeBw143do7BHOC3EbuwKYRZBfWLAhfX4EMmN9Susz2x0lwqVaFKEDf0AmxkFRNgAqnvH2wUS0sVdo2pEgQXXvaVqxJ9KHBwpoUaXPJA9OlISPeSlCAep2zPYiuyzGOTVOpY/oEQWNNCtlDDobxraJWhiJecr5MWpTqSqiepN0etaNy54zk5uxIgFTd14Tr2B4LASAcyrF+xpBfa5GVYCcXyEL5bk+Rfw+i666OaCuLz4rITixPpg7BXRFuY3hK9LXN/bhY52iYH3p/ISao0cXK2XRvtWvGhPK66nYDiJ2Gu7w1yE0ZZsoVc5Wrg765XV6V0CGrB6b6nvNbfCFKabGPtIwCpO979tcTL2pAmnuumxkzG+wVZkT0Zn3/iQz+zQovUu8ZnEwaxHGvG+JDSp/PiVkAoe3X9+iJy+b7+R6lCVnbWE3OdxBlQtQ65xLoQwK8AeL0ErtCcrsw9QxE1lbwZ+8ZS9LG3tCPkNDrSzKz2uY8EBbuC8agSb5lmL+v0b03q0RR97KSqsLvaiFdjTGdhDY8Fdrj5snsTGemhqppIQcx+tZf821s5gHgxhn23Ltyi9eBCiZ+w1JHwdS3Rk4Q43Sjw6gsZcK7AnA+wZaClNQnUKYV+PVJXOhRx3UeOd5OZ3z7jPMB11EOIDp3uCF2mcZbZglQBgKEwswZ2F89rZosnRvY0rtidD2Wfg7zKlS60res7+BeI7hyU5XgwJtThQ7c7iaJJ425DQNbgvmklCdaZeyiGD31lC7mIXnyUUV7tEK7gnI7PHhw8y1RKp9/nPYAJjng8H00c7McJZCZ16vBqsqmaF8Bap179oCdWolnDAfUBzQaZ/YRpM3EobIievJUQruCc24Is0pc3+rMZHtsJetdXBmuwca+2iWo+FhkCv8ew45qFM+97xq3bhtwVfKH5LfXT/1dmC20HTIpyz16ClFdqTEeOoC+zjeDsLlvA3jLm0CMyPWihDHubPoGklGuf8LFTVHPiOKSFkN0pO9x/i2BnukatT8vtn6p3Jy45lhfbUHFJValiGXqpVlyBF10mqQskOssQzlwJFmcwEen8U8j3RC8M6ZHscDX2VNKez9eCYUfH7jYeG92yvFdsXO0KraH4OCWHiuNhoj+iVZilUIQevbilUCLDjIYSOjxLzO8vsgSYcc2H0McS0MBOxeMwExtOoelqcsiL7ckb20HJKeujGQ5+nNeISzbw0wFdLClyajya0SuIoE6q2HpC/f8ORMkGu5dISMoeEQKHk7NZB+iB3WbCx1grsi5HiIIqX6Vz616KlJdBnbXNICDzsBU2IeYVWBqkqatZY41ti1yWrqunZFUuoGePU4pQCvODRhOiGx+ohibIi+2K0OIIUM+pszIgky6rWzsSPu4hBIPG72ZTl2Bqyk4atq/XuENmj9RS91KgPnPJUeimqLj5S7QI3j1YtK7QvZoAWsX7GrqZFDtmJ0BrpkK731AgJRPQHHKETn/1szH7UFm5BXoSTitdw5GNMfZ9I039d5MZ0CA6NvVZkXwxr3+LhOru3gXGw9oK2PA6kqqVFH8J2W0JyTLTtkV/dst5CewnayqXZ6ygBVOqsHHMf4HNoiZouWPuyAvtiWPtEEDGOWE/+tgQNmtRmS0jRDXH0CyHuI1mHuSBUDeEWuQQKx1ySQ0zIFGTuAndyO4699VBde63AvrCxV+s9xsO9knRpzjk3QjzYRm4s6OP5rD2EvWmS0H1Au95TOGLSMElz5CU8JWC7go5doWoH39m9jiuyL2LES1AiyZRL7Fz2GXod+tgxlYBH6K9L0LJNmixDL63eE776IqGcHGKC+8j/Ae0xDFYuw1fwiIm6QvtyhvYNQ0g85SUaoEtwmo5ol88hIQzoJL9F2zf0PPQSFVN8d4i9cCuYHL2EHk3ynNgLQxlprIX5Ww971RXYVzM/C4XDFHthKqHYcXbCtieeyzj6eKg/UpW5dxlHzzH0JvsNdSOH3KX+JnhBrdSmc/UJr+QCezm2hhhzrcC+GsoeuLPNHkeBjthqvbIsHW1IEpKIXzdmtGlXc4Xb5rO1lk1O4QghRKkLmEhSXGUvHdNfxmArtq/JRHvcxrD0aIvYGbSK4agw57Pz1cBx335RaMrss8bDdkv0AkUUPJpQP0Hih9qrjSkeUwsRBiA9e63gvprlOL0j9hh70X+l4A1VFZq1kP6po3Fk9UtySB/77Yxx0CjfOXqVBCGMbXIQVg22AVX7eCi7m70KdZ2WY60V2lcD7TFGL2E2OTBClr1VQstAaNS0Qj4tEdE4nbQEZL5vY70u1Us5zVGg18heg32dq4RQkvhQFSI6d6aqrsi+2vnZDK54Vo7UdwYZkhDTqpOWoEmNecG+xsl6QWBC7Z7oBazavHUc+AhhYi+NNZnlookm7rKXumL7Wh2FCU1FDoSCic1EKAU5Zn/2XLn3EVxsj10VczdOgrS33SD3wmP2ptkxj1+XjZeYkvBXL+1rRxxrrci+Gpk9tJehzUqIx3oWI+ydlD04FLpYvYRKaG5LAI6jlO4oHKmr+BzoBR1Enuo4qM+CO7BXahc2OeZagX01OvsivQlwmCtwZ6QNpxq32MJSOF4UQpjsyx287HuF5EH4fO+FqpWxZcsphBpYzBm9areet/QyHFt8zvbiFdhXK8cZ8wz9ajIN3uMMJWpLi1T1giaENpQoLC2OInwL7wXVFTktbSwJLZP3AqPJbqGNZlFw1V68Qns2OvuKYa7HQKgW2tkqAGTfOjAW9jKupM/jYJfvYwotd6XaDZx9gTIqO8iLgE0fA6HUh1Q8BQC6Vd70P6/Qno3OHsoBGeaCdrk5LQ6QPXN3CTENLtHb9UJ50F4dokm6o+HYatMw6dBemKePeTYcsc8quViiHa0QY64V2HMyLKH+HA2WsGiYUjxgC0c+8N+uJQx9aZGr7Q3H6oVdvIQu1S2FI2QxXrCHzCrNHaGYcDy2lBqkWo6pTWOuFddzdqSXqT7mEloX53ibXtLc4ljilRyHQ9riXF0CFJSIb9ktgcEbTxoXMHU2gH3nM90xDmzequ68Ma/AnsmZ2JOHdyFjFLuKA6twylxBi13MfvdfC1+UKFMtwQoT6ZayEduv2elxYLtFTnPrZSel4gWpGn17rciei9mpip3wvCwSOpZXvO5n12MzwV5EUp9H8gqhNLaQDaVck3KH0h5c8FDwvt5T1bCWZO7iqD3ye7E+H6ukjLlWXM9GjKOWyFzmRmg8NLItIaTGPKeEwmNx8XneOC7bxlFWpbftVL3ucWBHCdlgD21xeNgL6jMXqnKMB8to7LUiezbL7Dt4mAvHFTpFy0EXEF0zeimWwX4J114SjhWt+xpHRbg3ib26gtcZCUWgTbOFhm3zUtmd5GBgIs9eK7Rns/aS47IkFGqcQ5n+WrzE264j2FMlllT7M3toIc8F7dj3GN+0Guda2Iu672K5RJzaJQybu7mxi7RcFkdWYM8G2IP2yOM2FkXwxe7+36uxKVXFI7uY2UNunateGuaUY7ujbiyog5zghbXrHOZ8dprhwHDQaL54QFVWXC/BAaoy141XxjY9Z509oMRcsqd+3PxvlhC9p4/LSH21Q7pnrVcXxQbHvYZSeacM8wUFjeC3U6/GXiuwFyuzHwuf9uAVIE0ydRAALD8WaNexi80qCSEpnw0hwRAY37RcotKhPjoPOSqYl8e6BBnP1uFxmttBkxXai+HssVIgTDBBcUw0v7qPALRzfQlGvdife8G6BXkUjliYVu/ZSogBCmeBNniJx6YqrHgt5FZC+AoYd/GSrNBe7G4cKFbmlGPevy7m7F962YNMyh7XrPgNbb24MgUAqLK43oO9sFrAKRyRX+pjMCFyXxjryXEwmexexxXai91nPzr2u5SweLmx9FnRMrcSZojo/J2XsS/zGFiCuh74lm/j6CKE5AjHMRfGc6wq94XX3joO1o/kVY6yQnsxpD2wijxuo5rC4SWw02R+XQJEFTX5yl7odh7rJfRMXG+ZEsIXfUTne4S6PK9O9RL267OrVNWjJXfNnqzQXgxlD96R4gQTAfffQnvuo49jCi314swd5NBc3sKUS9Q+EPj9oRdo3uaNvej5hebMXsIX5bgzVdihVTxgLyuwFzaxi3rcHmKcPFYSnxto5bEZR7FtvPpqnHBE2tF8DHzLNwlhCW52utlYvpwyTSjRxrDv2bly6WN0jrlWXC92Nc6+OnxsS9gn7hykGuZGVfwIpXr1HWgLZY9dGCHfolTFA3R27PXxS36IS9JFexbt3ewKe9sK7MVsxsEi1ziRas/DzgyaHKNl+4yQ1hrii0vwBSwyJ/bQnmW6Zx00VnV6LA6+imUi+y4Od7ez49vRoislbCuyb0aMg7GQucy+aWzMzvpsqn0qaEi90hi8sYVQOL57YefvaWxL/t6MPQ7asjPEge0ycz67D0C1cKEb9/uNbQX2zajssYxjbrzsk39WqKp3bwsPJWGufbXDxe4Sbquwl3/DfPbv//l/oxqEQQ=='))
assert hashlib.sha256(ASSETS67['inputs/formal.json']).hexdigest()=='5d9c74662f1a0ecf5de144113cc06607c8cbe4d51ba3faf2fb62de8f98ed88b9'
ASSETS67['inputs/smoke.json']=zlib.decompress(base64.b64decode('eNqVkd1uwjAMRt/F17UUx2ma9CH2AhNCpWRTRFlZG9gQ6rvPDWM/wMXWq9g9dj6dnCA1w3NIy9Vx+dYP3RrqE+xi1ydU8hHUqvhRa6hpKiCTI9SPJ2hjOkIND00c+lWEAtr9mPptGKTZtG1CtoxOW/kTDuElzVNySqiNRWdV7ktFHlXlYVHAU9N1q6bdyIJxs0dWJPNauCjhfmWT1ss6vOeMMYVt3j3PaE+oShYgb5AExpl5t8xmaN6hlRdwDpYrsqg5Q0N43YfxDMp5htC7EopzpRipdGfwEIYxQC2RxyCX7bqYLhkFl0i7rkkCqKn4gyptHZIpb1WxwUqbL1VGQug7qrQ16MjdqtLfquhaVSXPQ+VFlauwdNW1KrbyVswXVUYjE99V5Tx69p+q2FTyxPw/VTQtpg9LPL+C'))
assert hashlib.sha256(ASSETS67['inputs/smoke.json']).hexdigest()=='3b1a91945a5a3e21f6964d9b32cf76a6fa4ba51c3369c8ca254ef2a474ba644b'
ASSETS67['inputs/reference.json']=zlib.decompress(base64.b64decode('...'))
assert hashlib.sha256(ASSETS67['inputs/reference.json']).hexdigest()=='ec105fa8c8296d8bde72ac24712f748cd7c9d8aa0bb192c995c1a9128a01cb0f'
ASSETS67['inputs/protocol.md']=zlib.decompress(base64.b64decode('eNptVk1vGzcQvedXDFAUSABJUZzEidGTmqRpCsM2EhdNewko7qzEiktuSa5k5ZT23HtP/Se99Z80P6B/oW+GK38AvdhaLjl88+bNm/2CtsezZ0TvTMfUxYY9Fb4qRN+9Oz8juozR04sYSjK50L17dLl2mfLQ9547DsV44quek5MHwqslu7CireMdNzOS4E+JTGj05zHRS85uFbghk8lQ4jz4MqO3PGSmsmbKgmPFgZMpMZELutrG1OGqfm0yI6qNoXWpm87n8xNgL/H20qOTo8+ffgXQMVgxacVlglCuOATJJdrNRDEVkzfAYGNq8oxeGbumXUy+oTSETH1iyYMe0kXillMCavr7D6IXa7YbcoXiUAi3ZmRA7x/+//qPRN/EIVFAJLBoN3miWUVwh9ePnz8hWqTirEdmR0RvQsM940+Q073zEcWQZBQZlp6TxPmZLfhxnCkGv6et8a4xhRG6cGqNxQvJkHNxnazbKGgQJ7GUKUQKpgxJ+TC4HkUDhQ0oigH1GAKx0HG9hmssa0gTjN9/xO+ipamEed4y0qnlvpMUNZHlvjJCEwD0/GvkJDK7k8nOlfVBJ3dj0OuLS6Lz5FYOtwuqkqL3KEillAyiJjaNIkyipkaJmYlmhfpVikN/54bv8ygQCT59KsAXlIPp8zoK9au+yOr0aH50PJ0/mR49hqzecu5jyAC7uHiD544bN3REp3Hl7IQabg0ELXF7D04nwm5oTGooc9o6UIjlQ22EYHoyPwEYKKaHaCCLDQci9BcnbO0ckLzactojqV8GVJN6k3E98LmgJ96VhIvQk9KA6BNqE/NHqedYjYsB3CjZtBCmib5FJaBCJePRHA0saBkrxpYB9Epb4KkdvEcnZ10SH0Dr4/y+tvPZq/eXHy7Pz08/vFicnn7+9BudqabcFu06BKuiSbzCGZS8mYjkhAPkpdWpBpPtmjsj/Xoh3RaHTAeK0QovpeagkPvpcj+V/1LkHAPSm8mJnEctQYjxoI7eJFyELDbaHWIY1cziUrSWvyLjfdzJK+JgfcxV/Q2A45kntASxTVTRJu6NA6mOvXrEFduhVKsyNUelsdwGIGTlUYlJWjlVT5XqyMYgD5KNNmQtszQlQQg1qHqE9UMjyNQA3RU4U08hH+MGWq7FQu9zxdTQcq97+xRXyXQA27aSL8VW1roe2USt803FDyVFmmBTDF2unyiHqt+axci5Np4Lg24ia4IwtBRDid4IAge32LpGcc3quOBqUJ2B6VmAMAm7g6b7+r60tUrzAVpIq1nXzqqOHoisbtlyzX+Q9rueFd4sxXnEZzVC3WPFisWV4b5ak1Ha/ZCsDBHd3zF6B12LWAEzbSleUq5tvwVl8lCdf0YLn6PwFNNofCmG1XQsBpLaUONasTc1SujcrMAIulaLFEX4ywipKOB8mGz5Zu7SW9KlW2GWXHYMcPdhrwEcvSb6iVMU6iFi4EEyYVXluFujuPilvVENUxtV+kMO3oTNYkvIHCIcrBXSTIsmRXmQqS13ZJarBBT0NPdsXeuszEwhuMq0nppiiCKGKYWhtPHYyLqQra/FVUDH0sgEBhk44LB/3A2HqLtmGJkY+EBinId94T3qMGSkjGphONtRpndOYY+qbGQWEzR2qL1YnmLRESHlY3FjCBGSFZkuYHNlDdesQyLTnW+RVuQzJpeRq9K7GPnCCLI6O0lMXMTOY8/JbdZ4O9TGqAQb7QDPZRyb41fI9RRFU4jb8GGMTXWO6IfUY3zrwOxZBvIE/7lBszw7fjQ/+vev3+llHJYyJk+efgnD52RlGACCAVT18Sk+A5h+0FteeIwQvAj724N0mUxAoy2HPbSpHy7/fPqTzuGR2TWH0HUuLeMgIHZrhztHGUINwsNNMSSDg6DGrPXTAiSBXqFnTBXWhC8yjJfGmVXAZ4qzYtu0CXEXKjFA+x8jIbQz'))
assert hashlib.sha256(ASSETS67['inputs/protocol.md']).hexdigest()=='77b3fb04ff1b2dd991edbc01c016eec9fb6190598caee1758aa5ac390c68a799'


In [ ]:
#@title  Check  384  The first free movement is true.  8B  Hints (none)  API）
EXPECTED67={'sources/01_shared.py': '2a1aea590c7ef67226080e533a1f295266ac284bce70a9d8d91de7ab344723f6', 'sources/02_receipts.py': 'cf9b8162ea774312e7288e329d8a05f0014b455a0bf545d2146c3547c98dbac0', 'sources/03_scoring.py': '01821d539b3baa7974e46b1ebae90018e2ff3a9e432c1aad4fa44da7c46121b4', 'sources/04_text_actor.py': '066cf25a6fada30d990921178e8b499981d6ba8787f47fb4bbf9f5d41839d0b8', 'sources/05_bridge.py': '1e529e416fc1bbfed236658d67d1c4f0e59229d7128f354c409f6b112747d4a1'}
NS67={"__name__":"text_bridge67"}
for name,source in SOURCES67.items():
    assert hashlib.sha256(source.encode()).hexdigest()==EXPECTED67[name]
    exec(compile(source,name,"exec"),NS67)
_formal=json.loads(ASSETS67["inputs/formal.json"]); _pilot=json.loads(ASSETS67["inputs/smoke.json"])
_ref=json.loads(ASSETS67["inputs/reference.json"])
assert [w["id"] for w in _formal["worlds"]]==[f"confirm-{i:05d}" for i in range(97,193)]
assert len(_pilot["worlds"])==2 and not set(_pilot["target_by_world"]) & set(_formal["target_by_world"])
checked=set()
for w in _formal["worlds"]:
    t=_formal["target_by_world"][w["id"]]
    for start in NS67["STARTS67"]:
        case,env,hist=NS67["start67"](w,t,start)
        texts=[]
        for label in NS67["LABELS67"]:
            key=f"{w['id']}/{t}-{label}-{start}"
            text=NS67["actor_prompt"](case,NS67["records_for"](w,t,label),hist)
            assert hashlib.sha256(text.encode()).hexdigest()==_ref["cells"][key]["first_prompt_sha256"]
            texts.append(text);checked.add(key)
        field='"attribute":"preview_item"'
        assert texts[0].count(field)==1 and texts[0].replace(field,'"attribute":"preferred_item"')==texts[1]
assert checked==set(_ref["cells"]) and len(checked)==384
print("384  An official first step reminder matches word by word. Checked on build  8B  Free action hint steps ",_ref["verified_historical_prompt_steps"])


In [ ]:
#@title  from  Colab Secrets  Initialize, export mode without key 
def bridge_client67():
    from google.colab import userdata
    hint=" Please.  Colab Secrets  Settings  OPENAI_API_KEY  ♪ And open this book ♪  notebook  Access rights. "
    try: key=userdata.get("OPENAI_API_KEY")
    except Exception: raise RuntimeError(hint) from None
    if not isinstance(key,str) or not key.strip():raise RuntimeError(hint)
    return NS67["OpenAI66"](key.strip())
CLIENT67=None if RUN_MODE=="export" else bridge_client67()


In [ ]:
import re
def v66_http_detail(response, *args, **kwargs):
    if response.status_code not in (400, 401, 403, 404, 422, 429):
        return
    try:
        error = response.json()["error"]
    except (ValueError, KeyError, TypeError):
        error = {}
    if not isinstance(error, dict):
        error = {}
    # Authentication errors sometimes echo part of the supplied credential.
    fields = ("type", "code", "param") if response.status_code == 401 else ("type", "code", "param", "message")
    parts = [f"{field}={error[field]}" for field in fields if isinstance(error.get(field), str) and error[field]]
    detail = " | ".join(parts) or " The server did not provide a visible error field. "
    sent = response.request.headers.get("Authorization", "").removeprefix("Bearer ")
    if sent:
        detail = detail.replace(sent, "[ Hide ]")
    detail = re.sub(r"(?i)\b(?:bearer\s+\S+|(?:sk-|sess-|hf_)[\w.*-]+)", "[ Hide ]", detail)
    print(f"OpenAI HTTP {response.status_code} | {' '.join(detail.split())[:600]}", flush=True)
    if len(sent.strip()) <= 1:
        print(" Last requested key is empty or only one character. Please update  OPENAI_API_KEY， Runs the client initialization again. ")
if CLIENT67 is not None and not any(getattr(h,"__name__","")=="v66_http_detail" for h in CLIENT67.session.hooks["response"]):
    CLIENT67.session.hooks["response"].append(v66_http_detail)


In [ ]:
#@title  Start or resume ( all  and  smoke  Call fee  API）
RESULT67=None
experiment=RUN67/"experiment"
if RUN_MODE=="export" and not (experiment/"manifest.json").is_file():raise RuntimeError(" No run to export ")
with NS67["writer67"](experiment,TAKEOVER_LOCK_ID) as owner:
    study=NS67["Bridge67"](experiment,ASSETS67,SOURCES67,CLIENT67,USD_BUDGET,RETRY_UNCERTAIN,owner)
    if RUN_MODE=="export":
        RESULT67=NS67["export67"](study,"full" if any(k.startswith("full/") for k in study.states) else "smoke")
    else:
        for stage in (("smoke","full") if RUN_MODE=="all" else ("smoke",)):
            try:
                study.run(stage)
            finally:
                RESULT67=NS67["export67"](study,stage)
            rep=RESULT67[0]
            if stage=="smoke":
                print("smoke   API  Usage ",rep["meter_stage"],"； Rough estimate of the formal stage  USD",round(48*rep["meter_stage"]["estimated_usd"],2))
            else:
                print(" Main indicator (percentage points) ",rep["primary"],"； Cumulative use of the current round ",rep["meter_all"])


In [ ]:
#@title  Verification  Drive  Package and release  runtime
from google.colab import runtime
NS67["finish67"](RESULT67,RUN67,MOUNT67,AUTO_DISCONNECT and RUN_MODE=="all",os.path.ismount,drive.flush_and_unmount,runtime.unassign)
